# Pythia-1B: обучение, perplexity и speed benchmark

В этом notebook используется только одна модель — `EleutherAI/pythia-1b`. Здесь нет sparse routing, Ocean или альтернативных архитектур. Notebook поддерживает streaming-обучение на больших англоязычных корпусах, оценку perplexity и измерение prefill/decode скорости.

Корпуса: [PG-19](https://github.com/google-deepmind/pg19), [Dolma](https://huggingface.co/datasets/allenai/dolma) и [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb).

In [8]:
# При необходимости установите зависимости отдельной командой:
# %pip install -U torch transformers datasets accelerate sentencepiece

import gc
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    TORCH_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    TORCH_DTYPE = torch.float16
else:
    TORCH_DTYPE = torch.float32

MODEL_ID = 'EleutherAI/pythia-1b'
print('device:', DEVICE)
print('dtype:', TORCH_DTYPE)
print('torch:', torch.__version__)

device: cuda
dtype: torch.bfloat16
torch: 2.14.0+cu126


In [9]:
# Безопасные значения по умолчанию для одной V100 32GB.
TRAIN_CONTEXT_LENGTH = 2_048
TRAIN_CHUNK_SIZE = 2_048
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
TRAIN_MAX_STEPS = 100
LEARNING_RATE = 2e-5
WARMUP_STEPS = 10
WEIGHT_DECAY = 0.1
GRADIENT_CLIP_NORM = 1.0

DATASET_NAME = 'pg19'
DATASET_SPLIT = 'train'
DATASET_CONFIG = None
MAX_TRAIN_DOCUMENTS = None
MAX_TRAIN_TOKENS = None

EVAL_MAX_DOCUMENTS = 4
EVAL_MAX_TOKENS = 32_768
SPEED_PROMPT_LENGTHS = (128, 512, 1_024, 2_048)
SPEED_NEW_TOKENS = 16
CHECKPOINT_DIR = Path('/home/froschin/work/llm/checkpoints/pythia-1b-long-context')

RUN_TRAINING = True
RUN_PPL = True
RUN_SPEED = True

print({
    'dataset': DATASET_NAME,
    'train_context_length': TRAIN_CONTEXT_LENGTH,
    'train_max_steps': TRAIN_MAX_STEPS,
    'run_training': RUN_TRAINING,
    'run_ppl': RUN_PPL,
    'run_speed': RUN_SPEED,
})

{'dataset': 'pg19', 'train_context_length': 2048, 'train_max_steps': 100, 'run_training': True, 'run_ppl': True, 'run_speed': True}


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
).to(DEVICE)
model.eval()

print('model:', MODEL_ID)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('native context:', model.config.max_position_embeddings)
print('vocab:', tokenizer.vocab_size)

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1476.05it/s]


model: EleutherAI/pythia-1b
parameters: 1011781632
native context: 2048
vocab: 50254


## Streaming datasets

`PG-19` используется как основной long-range language-modeling корпус. `Dolma` и `FineWeb` доступны в streaming-режиме и подходят для более масштабного continued pretraining. Данные не загружаются целиком в оперативную память.

In [11]:
def load_training_stream(name=DATASET_NAME, split=DATASET_SPLIT, config=None):
    name = name.lower()
    if name == 'pg19':
        return load_dataset('emozilla/pg19', split=split, streaming=True)
    if name == 'dolma':
        return load_dataset('allenai/dolma', split=split, streaming=True)
    if name == 'fineweb':
        config = config or 'sample-10BT'
        return load_dataset('HuggingFaceFW/fineweb', config, split=split, streaming=True)
    raise ValueError("name должен быть 'pg19', 'dolma' или 'fineweb'")

def extract_text(record):
    for key in ('text', 'content', 'document'):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value
    raise KeyError(f'Не найден текстовый field в записи: {tuple(record)}')

def token_block_stream(dataset, tokenizer, block_size=TRAIN_CONTEXT_LENGTH, max_documents=None, max_tokens=None):
    buffer = []
    documents = 0
    tokens_seen = 0
    for record in dataset:
        if max_documents is not None and documents >= max_documents:
            break
        ids = tokenizer(extract_text(record), add_special_tokens=False).input_ids
        documents += 1
        if max_tokens is not None:
            remaining = max_tokens - tokens_seen
            if remaining <= 0:
                break
            ids = ids[:remaining]
        buffer.extend(ids)
        tokens_seen += len(ids)
        while len(buffer) >= block_size:
            yield torch.tensor(buffer[:block_size], dtype=torch.long)
            del buffer[:block_size]
        if max_tokens is not None and tokens_seen >= max_tokens:
            break

def make_train_stream():
    dataset = load_training_stream(DATASET_NAME, DATASET_SPLIT, DATASET_CONFIG)
    return token_block_stream(
        dataset, tokenizer, block_size=TRAIN_CHUNK_SIZE,
        max_documents=MAX_TRAIN_DOCUMENTS, max_tokens=MAX_TRAIN_TOKENS,
    )

print('dataset loader ready:', DATASET_NAME)

dataset loader ready: pg19


In [12]:
@torch.inference_mode()
def evaluate_ppl_on_stream(model, dataset, max_documents=4, max_tokens=32_768, block_size=2_048):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    blocks = token_block_stream(
        dataset, tokenizer, block_size=block_size,
        max_documents=max_documents, max_tokens=max_tokens,
    )
    start_time = time.perf_counter()
    for block in blocks:
        input_ids = block[:-1].unsqueeze(0).to(DEVICE)
        targets = block[1:].unsqueeze(0).to(DEVICE)
        logits = model(input_ids, use_cache=False).logits
        total_nll += F.cross_entropy(
            logits.float().reshape(-1, model.config.vocab_size),
            targets.reshape(-1), reduction='sum'
        ).item()
        total_tokens += targets.numel()
    elapsed = time.perf_counter() - start_time
    mean_nll = total_nll / max(total_tokens, 1)
    return {
        'mode': 'windowed_ppl',
        'documents': max_documents,
        'tokens': total_tokens,
        'block_size': block_size,
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': total_tokens / max(elapsed, 1e-9),
    }

def run_ppl_benchmark():
    dataset = load_training_stream(DATASET_NAME, 'validation', DATASET_CONFIG)
    result = evaluate_ppl_on_stream(
        model, dataset, max_documents=EVAL_MAX_DOCUMENTS,
        max_tokens=EVAL_MAX_TOKENS, block_size=TRAIN_CONTEXT_LENGTH,
    )
    print('--- Pythia PPL benchmark ---')
    print(result)
    return result

if RUN_PPL:
    ppl_result = run_ppl_benchmark()

--- Pythia PPL benchmark ---
{'mode': 'windowed_ppl', 'documents': 4, 'tokens': 32752, 'block_size': 2048, 'mean_nll': 3.003555842009648, 'perplexity': 20.157085050696296, 'seconds': 13.052280526142567, 'tokens_per_second': 2509.2932943327896}


In [13]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def clear_gpu_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

@torch.inference_mode()
def benchmark_speed(model, prompt_ids, prompt_length, new_tokens=16, allow_untrained_context=False):
    native_context = int(model.config.max_position_embeddings)
    if prompt_length > native_context and not allow_untrained_context:
        raise ValueError(
            f'prompt_length={prompt_length} > native context={native_context}; '
            'set allow_untrained_context=True explicitly'
        )
    source = prompt_ids.flatten().to(DEVICE)
    prompt = source.repeat(math.ceil(prompt_length / source.numel()))[:prompt_length].unsqueeze(0)
    model.eval()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    synchronize()
    prefill_start = time.perf_counter()
    outputs = model(prompt, use_cache=True)
    synchronize()
    prefill_seconds = time.perf_counter() - prefill_start
    next_token = outputs.logits[:, -1:, :].argmax(dim=-1)

    synchronize()
    decode_start = time.perf_counter()
    for _ in range(new_tokens):
        outputs = model(next_token, past_key_values=outputs.past_key_values, use_cache=True)
        next_token = outputs.logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start
    result = {
        'prompt_length': prompt_length,
        'new_tokens': new_tokens,
        'prefill_seconds': prefill_seconds,
        'prefill_tokens_per_second': prompt_length / max(prefill_seconds, 1e-9),
        'decode_seconds': decode_seconds,
        'decode_tokens_per_second': new_tokens / max(decode_seconds, 1e-9),
        'total_seconds': prefill_seconds + decode_seconds,
        'model': MODEL_ID,
    }
    if DEVICE.type == 'cuda':
        result['peak_cuda_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
        result['peak_cuda_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
    return result

def run_speed_benchmark():
    dataset = load_training_stream(DATASET_NAME, 'validation', DATASET_CONFIG)
    first_record = next(iter(dataset))
    prompt_ids = torch.tensor(
        tokenizer(extract_text(first_record), add_special_tokens=False).input_ids,
        dtype=torch.long,
    )
    results = []
    for length in SPEED_PROMPT_LENGTHS:
        result = benchmark_speed(
            model, prompt_ids, length, new_tokens=SPEED_NEW_TOKENS,
            allow_untrained_context=False,
        )
        results.append(result)
        print(result)
    return results

if RUN_SPEED:
    speed_results = run_speed_benchmark()

{'prompt_length': 128, 'new_tokens': 16, 'prefill_seconds': 0.18880924489349127, 'prefill_tokens_per_second': 677.9329056276125, 'decode_seconds': 0.3514548419043422, 'decode_tokens_per_second': 45.52505213274264, 'total_seconds': 0.5402640867978334, 'model': 'EleutherAI/pythia-1b', 'peak_cuda_allocated_gib': 1.9244780540466309, 'peak_cuda_reserved_gib': 4.6328125}
{'prompt_length': 512, 'new_tokens': 16, 'prefill_seconds': 0.119150061160326, 'prefill_tokens_per_second': 4297.102284413121, 'decode_seconds': 0.18823348684236407, 'decode_tokens_per_second': 85.00081610558053, 'total_seconds': 0.3073835480026901, 'model': 'EleutherAI/pythia-1b', 'peak_cuda_allocated_gib': 2.0151724815368652, 'peak_cuda_reserved_gib': 4.6328125}
{'prompt_length': 1024, 'new_tokens': 16, 'prefill_seconds': 0.2070185139309615, 'prefill_tokens_per_second': 4946.417499361884, 'decode_seconds': 0.1886015369091183, 'decode_tokens_per_second': 84.83493964161036, 'total_seconds': 0.3956200508400798, 'model': 'Eleu

In [14]:
def train_pythia_streaming(model, train_stream, max_steps=TRAIN_MAX_STEPS):
    model.train()
    model.config.use_cache = False
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=max_steps
    )
    use_scaler = DEVICE.type == 'cuda' and TORCH_DTYPE == torch.float16
    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)
    autocast_enabled = DEVICE.type == 'cuda'
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    start_time = time.perf_counter()

    for step in range(max_steps):
        block = next(train_stream).unsqueeze(0).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=TORCH_DTYPE, enabled=autocast_enabled):
            loss = model(input_ids=block, labels=block).loss
            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
        if use_scaler:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            if use_scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            if use_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item()
        if (step + 1) % 10 == 0:
            print({
                'step': step + 1,
                'loss': running_loss / 10.0,
                'ppl': math.exp(min(running_loss / 10.0, 20.0)),
                'lr': scheduler.get_last_lr()[0],
                'seconds': time.perf_counter() - start_time,
            })
            running_loss = 0.0

    model.config.use_cache = True
    model.eval()
    return model

def run_training():
    train_stream = make_train_stream()
    trained_model = train_pythia_streaming(model, train_stream)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    trained_model.save_pretrained(CHECKPOINT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(CHECKPOINT_DIR)
    print('checkpoint saved:', CHECKPOINT_DIR)
    return trained_model

if RUN_TRAINING:
    model = run_training()

/tmp/ipykernel_2833953/1363960710.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)


{'step': 10, 'loss': 0.9276154220104218, 'ppl': 2.528472642972765, 'lr': 2.0000000000000003e-06, 'seconds': 51.43326001800597}
{'step': 20, 'loss': 1.438740360736847, 'ppl': 4.2153826098923135, 'lr': 4.000000000000001e-06, 'seconds': 67.68184989201836}
{'step': 30, 'loss': 1.5685736179351806, 'ppl': 4.7997969643966405, 'lr': 6e-06, 'seconds': 84.01507993601263}


KeyboardInterrupt: 

## Ограничения

1. Исходный Pythia-1B обучен с native context 2048 токенов. Continued pretraining на длинных последовательностях требует увеличения `TRAIN_CONTEXT_LENGTH`, positional encoding scaling и существенного бюджета вычислений.
2. `evaluate_ppl_on_stream` использует независимые блоки фиксированного размера. Это корректный baseline PPL, но не проверка дальних зависимостей между блоками.
3. Полное обучение Pythia-1B с AdamW требует значительной GPU-памяти. На V100 следует начинать с небольшого `TRAIN_MAX_STEPS`, включённого gradient checkpointing и короткого контекста.
4. Dolma и FineWeb — огромные streaming-корпуса. Их нельзя запускать без ограничения `TRAIN_MAX_STEPS` или `MAX_TRAIN_TOKENS`.